# 08 · PyIceberg: creating and writing tables

**Time:** ~30 min · **You will learn:** create tables, append, overwrite, delete, upsert and evolve the schema from Python,
then read the result from Spark.

In [ ]:
import pyarrow as pa
from datetime import datetime
from pyiceberg.catalog import load_catalog

catalog = load_catalog("default")
if ("course",) not in catalog.list_namespaces():
    catalog.create_namespace("course")
catalog.drop_table("course.customers") if catalog.table_exists("course.customers") else None

## 1. Create a table from a PyArrow schema

In [ ]:
schema = pa.schema([
    ("customer_id", pa.int64()),
    ("name", pa.string()),
    ("city", pa.string()),
    ("signup_ts", pa.timestamp("us")),
])
customers = catalog.create_table("course.customers", schema=schema)
print(customers.schema())

## 2. Append data
Data is passed as a PyArrow table. Each `append` is one atomic commit.

In [ ]:
batch1 = pa.table({
    "customer_id": [1, 2, 3],
    "name": ["Alice", "Bob", "Carol"],
    "city": ["Berlin", "Paris", "Madrid"],
    "signup_ts": [datetime(2024, 1, 5), datetime(2024, 1, 9), datetime(2024, 2, 1)],
}, schema=schema)

customers.append(batch1)
print(customers.scan().to_pandas())

## 3. Upsert
`upsert` updates rows that match the join columns and inserts the rest, like a `MERGE`.

In [ ]:
changes = pa.table({
    "customer_id": [2, 4],
    "name": ["Bob", "Dave"],
    "city": ["Lyon", "Rome"],              # Bob moved, Dave is new
    "signup_ts": [datetime(2024, 1, 9), datetime(2024, 2, 20)],
}, schema=schema)

result = customers.upsert(changes, join_cols=["customer_id"])
print("updated:", result.rows_updated, "inserted:", result.rows_inserted)
print(customers.scan().to_pandas().sort_values("customer_id"))

## 4. Overwrite and delete with a filter

In [ ]:
customers.delete(delete_filter="customer_id = 4")

customers.overwrite(
    pa.table({"customer_id": [3], "name": ["Carol"], "city": ["Valencia"],
              "signup_ts": [datetime(2024, 2, 1)]}, schema=schema),
    overwrite_filter="customer_id = 3",
)
print(customers.scan().to_pandas().sort_values("customer_id"))

## 5. Evolve the schema

In [ ]:
from pyiceberg.types import StringType

with customers.update_schema() as update:
    update.add_column("country", StringType(), doc="ISO country name")

customers = catalog.load_table("course.customers")
print(customers.schema())

Existing rows read `country` as null. New writes can include it. After a schema change we pass a PyArrow table that
has the **new** schema:

In [ ]:
new_schema = pa.schema([*schema, pa.field("country", pa.string())])
customers.append(pa.table({
    "customer_id": [5], "name": ["Erin"], "city": ["Vienna"],
    "signup_ts": [datetime(2024, 3, 3)], "country": ["Austria"],
}, schema=new_schema))
print(customers.scan().to_pandas().sort_values("customer_id"))

## 6. Partition a table
Partition specs can be added with `update_spec`. We use `month(signup_ts)`:

In [ ]:
from pyiceberg.transforms import MonthTransform

with customers.update_spec() as spec_update:
    spec_update.add_field("signup_ts", MonthTransform(), "signup_month")

print(catalog.load_table("course.customers").spec())

## 7. Read the PyIceberg table from Spark
This is the interoperability payoff: same files, same catalog, different engine.

In [ ]:
from helpers import get_spark

spark = get_spark("08-read-from-spark")
spark.table("demo.course.customers").orderBy("customer_id").show()
spark.sql("SELECT committed_at, operation FROM demo.course.customers.snapshots ORDER BY committed_at").show(truncate=False)

### Your turn

Create a table `course.products` (`product_id` long, `name` string, `price` double) with PyIceberg,
append two products, **upsert** a price change for one of them, then confirm from **Spark** that the new price is visible.

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
p_schema = pa.schema([("product_id", pa.int64()), ("name", pa.string()), ("price", pa.float64())])
if catalog.table_exists("course.products"):
    catalog.drop_table("course.products")
products = catalog.create_table("course.products", schema=p_schema)
products.append(pa.table({"product_id": [1, 2], "name": ["keyboard", "mouse"], "price": [49.0, 19.5]}, schema=p_schema))
products.upsert(pa.table({"product_id": [2], "name": ["mouse"], "price": [24.0]}, schema=p_schema), join_cols=["product_id"])

price = spark.sql("SELECT price FROM demo.course.products WHERE product_id = 2").collect()[0][0]
assert price == 24.0
```

</details>

### Recap
* `create_table`, `append`, `overwrite(..., overwrite_filter=...)`, `delete(...)`, `upsert(..., join_cols=[...])` cover the everyday writes.
* `update_schema()` / `update_spec()` evolve tables transactionally.
* Everything PyIceberg writes is immediately readable from Spark (and vice versa).

Next: **09 · Maintenance and capstone project**.